# Customer Intelligence Dataset
This notebook integrates the outputs of the ChurnGuard pipeline into a unified customer-level dataset.
The final dataset combines:
- Customer information
- XGBoost churn probability
- Customer risk level
- Estimated revenue at risk
- Rule-based retention recommendations
The resulting dataset will be used as the primary data source for the ChurnGuard Streamlit dashboard.

In [1]:
import pandas as pd

In [2]:
raw_data = pd.read_csv(
    "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

print("Raw dataset shape:", raw_data.shape)

Raw dataset shape: (7043, 21)


In [3]:
xgb_predictions = pd.read_csv(
    "../data/processed/xgboost_predictions.csv"
)

print("XGBoost predictions shape:", xgb_predictions.shape)
print("\nColumns:")
print(xgb_predictions.columns.tolist())

XGBoost predictions shape: (1409, 3)

Columns:
['Actual_Churn', 'Predicted_Churn', 'Churn_Probability']


In [4]:
revenue_risk = pd.read_csv(
    "../reports/revenue_at_risk.csv"
)

print("Revenue risk shape:", revenue_risk.shape)
print("\nColumns:")
print(revenue_risk.columns.tolist())

Revenue risk shape: (1409, 6)

Columns:
['tenure', 'Contract', 'MonthlyCharges', 'Churn_Probability', 'Risk_Level', 'Estimated_Revenue_at_Risk']


In [5]:
X = raw_data.drop(columns=["Churn"])

y = raw_data["Churn"].map({
    "No": 0,
    "Yes": 1
})

In [6]:
from sklearn.model_selection import train_test_split

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_test shape:", X_test_raw.shape)
print("y_test shape:", y_test_raw.shape)

X_test shape: (1409, 20)
y_test shape: (1409,)


In [7]:
import sys
import os
project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.recommendation_engine import generate_retention_plan

print("Recommendation engine loaded!")

Recommendation engine loaded!


In [8]:
all_recommendations = []

for i in range(len(X_test_raw)):

    customer = X_test_raw.iloc[i]

    churn_probability = (
        xgb_predictions.iloc[i]["Churn_Probability"]
    )

    retention_plan = generate_retention_plan(
        customer,
        churn_probability
    )

    all_recommendations.append(
        " | ".join(retention_plan)
    )

print(
    "Number of recommendation records:",
    len(all_recommendations)
)

Number of recommendation records: 1409


In [9]:
recommendations_all = pd.DataFrame({
    "Customer_Index": range(len(X_test_raw)),
    "Churn_Probability": xgb_predictions["Churn_Probability"],
    "Recommendations": all_recommendations
})

print("Recommendations shape:", recommendations_all.shape)

Recommendations shape: (1409, 3)


In [10]:
recommendations_all.head()

,Customer_Index,Churn_Probability,Recommendations
0,0,0.017472,Customer is currently low risk; continue regul...
1,1,0.851540,Customer is high risk; prioritize for retentio...
2,2,0.054626,Customer is currently low risk; continue regul...
3,3,0.304251,Customer is currently low risk; continue regul...
4,4,0.012186,Customer is currently low risk; continue regul...


In [11]:
print(
    "Missing recommendations:",
    recommendations_all["Recommendations"].isna().sum()
)

Missing recommendations: 0


In [12]:
recommendations_all.to_csv(
    "../reports/retention_recommendations.csv",
    index=False
)

print("Updated retention_recommendations.csv successfully.")

Updated retention_recommendations.csv successfully.


In [13]:
customer_intelligence = X_test_raw.copy()

print(
    "Starting customer intelligence shape:",
    customer_intelligence.shape
)

Starting customer intelligence shape: (1409, 20)


In [14]:
customer_intelligence["Churn_Probability"] = (
    xgb_predictions["Churn_Probability"].values
)

In [15]:
customer_intelligence["Predicted_Churn"] = (
    xgb_predictions["Predicted_Churn"].values
)

In [16]:
def get_risk_level(probability):

    if probability < 0.40:
        return "Low"

    elif probability < 0.70:
        return "Medium"

    else:
        return "High"

In [17]:
customer_intelligence["Risk_Level"] = (
    customer_intelligence["Churn_Probability"]
    .apply(get_risk_level)
)

In [18]:
customer_intelligence["Estimated_Revenue_at_Risk"] = (
    customer_intelligence["MonthlyCharges"]
    * customer_intelligence["Churn_Probability"]
)

In [19]:
customer_intelligence["Recommendations"] = (
    all_recommendations
)

In [20]:
customer_intelligence["Customer_Index"] = range(
    len(customer_intelligence)
)

In [21]:
customer_intelligence = customer_intelligence[
    [
        "Customer_Index",
        "tenure",
        "Contract",
        "MonthlyCharges",
        "Churn_Probability",
        "Risk_Level",
        "Estimated_Revenue_at_Risk",
        "Recommendations"
    ]
]

In [22]:
print(
    "Final dataset shape:",
    customer_intelligence.shape
)

Final dataset shape: (1409, 8)


In [23]:
print(
    customer_intelligence.columns.tolist()
)

['Customer_Index', 'tenure', 'Contract', 'MonthlyCharges', 'Churn_Probability', 'Risk_Level', 'Estimated_Revenue_at_Risk', 'Recommendations']


In [24]:
print(
    customer_intelligence.isnull().sum()
)

Customer_Index               0
tenure                       0
Contract                     0
MonthlyCharges               0
Churn_Probability            0
Risk_Level                   0
Estimated_Revenue_at_Risk    0
Recommendations              0
dtype: int64


In [25]:
print(
    customer_intelligence["Risk_Level"].value_counts()
)

Risk_Level
Low       899
Medium    287
High      223
Name: count, dtype: int64


In [26]:
total_revenue_at_risk = (
    customer_intelligence[
        "Estimated_Revenue_at_Risk"
    ].sum()
)

print(
    "Total Estimated Revenue at Risk:",
    total_revenue_at_risk
)

Total Estimated Revenue at Risk: 32572.03863592467


In [27]:
def check_recommendation_risk(row):

    probability = row["Churn_Probability"]
    recommendation = row["Recommendations"].lower()

    if probability < 0.40:
        expected = "low risk"

    elif probability < 0.70:
        expected = "medium risk"

    else:
        expected = "high risk"

    return expected in recommendation

In [28]:
risk_check = customer_intelligence.apply(
    check_recommendation_risk,
    axis=1
)

print(
    "Risk/recommendation mismatches:",
    (~risk_check).sum()
)

Risk/recommendation mismatches: 0


In [29]:
customer_intelligence.head(10)

,Customer_Index,tenure,Contract,MonthlyCharges,Churn_Probability,Risk_Level,Estimated_Revenue_at_Risk,Recommendations
437,0,72,Two year,114.05,0.017472,Low,1.992680,Customer is currently low risk; continue regul...
2280,1,8,Month-to-month,100.15,0.851540,High,85.281771,Customer is high risk; prioritize for retentio...
2235,2,41,One year,78.35,0.054626,Low,4.279973,Customer is currently low risk; continue regul...
4460,3,18,Month-to-month,78.20,0.304251,Low,23.792413,Customer is currently low risk; continue regul...
3761,4,72,Two year,82.65,0.012186,Low,1.007162,Customer is currently low risk; continue regul...
5748,5,21,Month-to-month,99.85,0.719245,High,71.816663,Customer is high risk; prioritize for retentio...
3568,6,21,Month-to-month,99.15,0.520119,Medium,51.569812,Customer is medium risk; consider targeted ret...
2976,7,19,Month-to-month,24.10,0.100247,Low,2.415949,Customer is currently low risk; continue regul...
5928,8,61,Two year,19.75,0.010632,Low,0.209973,Customer is currently low risk; continue regul...
1639,9,17,Month-to-month,45.05,0.552585,Medium,24.893944,Customer is medium risk; consider targeted ret...


In [30]:
customer_intelligence.tail()

,Customer_Index,tenure,Contract,MonthlyCharges,Churn_Probability,Risk_Level,Estimated_Revenue_at_Risk,Recommendations
5143,1404,49,One year,87.20,0.052499,Low,4.577922,Customer is currently low risk; continue regul...
4439,1405,28,Two year,20.30,0.009612,Low,0.195120,Customer is currently low risk; continue regul...
3857,1406,5,Month-to-month,20.65,0.231932,Low,4.789405,Customer is currently low risk; continue regul...
4758,1407,56,Two year,19.70,0.035547,Low,0.700278,Customer is currently low risk; continue regul...
5613,1408,72,Two year,70.45,0.005528,Low,0.389482,Customer is currently low risk; continue regul...


In [31]:
print(
    "Customers with recommendations:",
    customer_intelligence["Recommendations"].notna().sum()
)

print(
    "Total customers:",
    len(customer_intelligence)
)

Customers with recommendations: 1409
Total customers: 1409


In [32]:
customer_intelligence.to_csv(
    "../reports/customer_intelligence.csv",
    index=False
)

print(
    "customer_intelligence.csv saved successfully!"
)

customer_intelligence.csv saved successfully!


In [33]:
final_check = pd.read_csv(
    "../reports/customer_intelligence.csv"
)

print(
    "Saved file shape:",
    final_check.shape
)

print(
    "\nSaved file columns:"
)

print(
    final_check.columns.tolist()
)

Saved file shape: (1409, 8)

Saved file columns:
['Customer_Index', 'tenure', 'Contract', 'MonthlyCharges', 'Churn_Probability', 'Risk_Level', 'Estimated_Revenue_at_Risk', 'Recommendations']


In [34]:
print(
    final_check.isnull().sum()
)

Customer_Index               0
tenure                       0
Contract                     0
MonthlyCharges               0
Churn_Probability            0
Risk_Level                   0
Estimated_Revenue_at_Risk    0
Recommendations              0
dtype: int64


In [35]:
final_check.head()

,Customer_Index,tenure,Contract,MonthlyCharges,Churn_Probability,Risk_Level,Estimated_Revenue_at_Risk,Recommendations
0,0,72,Two year,114.05,0.017472,Low,1.992680,Customer is currently low risk; continue regul...
1,1,8,Month-to-month,100.15,0.851540,High,85.281771,Customer is high risk; prioritize for retentio...
2,2,41,One year,78.35,0.054626,Low,4.279973,Customer is currently low risk; continue regul...
3,3,18,Month-to-month,78.20,0.304251,Low,23.792413,Customer is currently low risk; continue regul...
4,4,72,Two year,82.65,0.012186,Low,1.007162,Customer is currently low risk; continue regul...
